In [1]:
import torch

In [2]:
import plotly.express as px 

c:\Users\danny\anaconda3\envs\py310\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
a = torch.tensor([1,2,3,4,5,6])
b = torch.tensor([1,1,1])

In [71]:
b_padded = torch.nn.functional.pad(b, (0, a.size(0) - b.size(0)))

a_f = torch.fft.fft(a)
b_f = torch.fft.fft(b_padded)

In [76]:
# Cross-correlation using FFT: corr(a, b) = IFFT(FFT(a) * conj(FFT(b)))
# No flip needed - just conjugate the kernel's FFT

c = torch.fft.ifft(a_f * b_f.conj())
c_real = c.real

# Valid correlation output (same as mode='valid' in numpy)
c_real_valid = c_real[:a.size(0) - b.size(0) + 1]

px.line(c_real_valid, title="Cross-correlation via FFT")

In [79]:
# Cross-correlation using FFT with time-domain reversal
# The property is: FFT(x[-n mod N]) = conj(FFT(x))
# This requires CIRCULAR reversal: keep x[0], reverse x[1:N]

def circular_reverse(x):
    return torch.cat([x[:1], torch.flip(x[1:], dims=[0])])

b_rev_padded = circular_reverse(b_padded)
b_rev_f = torch.fft.fft(b_rev_padded)

c = torch.fft.ifft(a_f * b_rev_f)
c_real = c.real

c_real_valid = c_real[:a.size(0) - b.size(0) + 1]

px.line(c_real_valid, title="Cross-correlation via FFT (circular reversal)")

In [78]:
# The correct relationship is: FFT(x[-n]) = conj(FFT(x))
# where x[-n] is CIRCULAR reversal, not simple flip
# Circular reversal: keep x[0], reverse x[1:N]

def circular_reverse(x):
    """Circular time reversal: x[0] stays, x[1:] reversed"""
    return torch.cat([x[:1], torch.flip(x[1:], dims=[0])])

# Method 1: conj
c1 = torch.fft.ifft(a_f * b_f.conj())
c1_valid = c1.real[:a.size(0) - b.size(0) + 1]

# Method 2: circular reversal
b_circular_rev = circular_reverse(b_padded)
b_rev_f = torch.fft.fft(b_circular_rev)
c2 = torch.fft.ifft(a_f * b_rev_f)
c2_valid = c2.real[:a.size(0) - b.size(0) + 1]

print("b_padded:        ", b_padded)
print("circular_reverse:", b_circular_rev)
print()
print("Method 1 (conj):          ", c1_valid)
print("Method 2 (circular flip): ", c2_valid)
print("Are they equal?", torch.allclose(c1_valid, c2_valid))

b_padded:         tensor([1, 1, 1, 0, 0, 0])
circular_reverse: tensor([1, 0, 0, 0, 1, 1])

Method 1 (conj):           tensor([ 6.,  9., 12., 15.])
Method 2 (circular flip):  tensor([ 6.,  9., 12., 15.])
Are they equal? True


In [82]:
torch.fft.ifft(a_f.conj())

tensor([1.+0.j, 6.+0.j, 5.-0.j, 4.+0.j, 3.+0.j, 2.+0.j])